In [ ]:
import os
import json
from pathlib import Path
from collections import defaultdict

print("Hunting for the dataset files...\n")

# 1. Automatically search for the JSON files (looking up two folder levels to catch everything)
train_json_path = next(Path("../..").rglob("instances_train_trashcan.json"), None)
val_json_path = next(Path("../..").rglob("instances_val_trashcan.json"), None)

if not train_json_path or not val_json_path:
    print("ERROR: Could not find the JSON files. Please verify the dataset is downloaded.")
else:
    print(f"SUCCESS! Found Train JSON: {train_json_path}")
    print(f"SUCCESS! Found Val JSON: {val_json_path}\n")

    # 2. Setup output directories safely inside our Git repo
    output_dir_train = Path("../data/yolo_labels/train")
    output_dir_val = Path("../data/yolo_labels/val")
    output_dir_train.mkdir(parents=True, exist_ok=True)
    output_dir_val.mkdir(parents=True, exist_ok=True)

    # --- PROCESS TRAINING DATA ---
    print("Converting Training Labels...")
    with open(train_json_path, "r") as f:
        train_data = json.load(f)

    # Class Mapping (COCO to YOLO)
    coco_to_yolo_id = {cat['id']: idx for idx, cat in enumerate(train_data['categories'])}
    
    train_image_dict = {img['id']: img for img in train_data['images']}
    train_anns_by_image = defaultdict(list)
    for ann in train_data['annotations']:
        train_anns_by_image[ann['image_id']].append(ann)

    for img_id, img_info in train_image_dict.items():
        txt_name = os.path.splitext(os.path.basename(img_info['file_name']))[0] + ".txt"
        with open(output_dir_train / txt_name, "w") as f:
            for ann in train_anns_by_image.get(img_id, []):
                x_min, y_min, w, h = ann['bbox']
                # Math + Bounds checking in one line
                x_c = max(0.0, min(1.0, (x_min + w/2) / img_info['width']))
                y_c = max(0.0, min(1.0, (y_min + h/2) / img_info['height']))
                norm_w = max(0.0, min(1.0, w / img_info['width']))
                norm_h = max(0.0, min(1.0, h / img_info['height']))
                f.write(f"{coco_to_yolo_id[ann['category_id']]} {x_c:.6f} {y_c:.6f} {norm_w:.6f} {norm_h:.6f}\n")

    # --- PROCESS VALIDATION DATA ---
    print("Converting Validation Labels...")
    with open(val_json_path, "r") as f:
        val_data = json.load(f)

    val_image_dict = {img['id']: img for img in val_data['images']}
    val_anns_by_image = defaultdict(list)
    for ann in val_data['annotations']:
        val_anns_by_image[ann['image_id']].append(ann)

    for img_id, img_info in val_image_dict.items():
        txt_name = os.path.splitext(os.path.basename(img_info['file_name']))[0] + ".txt"
        with open(output_dir_val / txt_name, "w") as f:
            for ann in val_anns_by_image.get(img_id, []):
                x_min, y_min, w, h = ann['bbox']
                # Math + Bounds checking
                x_c = max(0.0, min(1.0, (x_min + w/2) / img_info['width']))
                y_c = max(0.0, min(1.0, (y_min + h/2) / img_info['height']))
                norm_w = max(0.0, min(1.0, w / img_info['width']))
                norm_h = max(0.0, min(1.0, h / img_info['height']))
                f.write(f"{coco_to_yolo_id[ann['category_id']]} {x_c:.6f} {y_c:.6f} {norm_w:.6f} {norm_h:.6f}\n")

    print("\nSUCCESS! All labels converted and stored safely in your project folder.")

In [ ]:
import os
import json
import yaml
import shutil
from pathlib import Path

print("Reorganizing folders for YOLOv8...")

base_dir = Path("../data")

# 1. Create standard YOLO directories
images_dir = base_dir / "images"
labels_dir = base_dir / "labels"
images_dir.mkdir(exist_ok=True)
labels_dir.mkdir(exist_ok=True)

# 2. Instantly move the image folders to data/images/
old_train_imgs = base_dir / "dataset" / "instance_version" / "train"
old_val_imgs = base_dir / "dataset" / "instance_version" / "val"

if old_train_imgs.exists():
    shutil.move(str(old_train_imgs), str(images_dir / "train"))
if old_val_imgs.exists():
    shutil.move(str(old_val_imgs), str(images_dir / "val"))

# 3. Instantly move the label folders to data/labels/
old_train_lbls = base_dir / "yolo_labels" / "train"
old_val_lbls = base_dir / "yolo_labels" / "val"

if old_train_lbls.exists():
    shutil.move(str(old_train_lbls), str(labels_dir / "train"))
if old_val_lbls.exists():
    shutil.move(str(old_val_lbls), str(labels_dir / "val"))

print("Folders reorganized successfully!")

# 4. Auto-generate the dataset.yaml file
json_path = next(Path("../..").rglob("instances_train_trashcan.json"), None)
with open(json_path, "r") as f:
    data = json.load(f)

# Extract class names in the exact order needed (0 to 15)
class_names = [cat['name'] for cat in data['categories']]

yaml_content = {
    'path': '../data',          # Root dataset folder
    'train': 'images/train',    # Train images (relative to path)
    'val': 'images/val',        # Val images (relative to path)
    'nc': len(class_names),
    'names': class_names
}

yaml_path = base_dir / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"\nSUCCESS! dataset.yaml generated at {yaml_path}")
print("Phase 1, Week 1 is officially complete!")

In [ ]:
import os
import json
import shutil
from pathlib import Path

print("Activating File Fixer: Locating all hidden images...")

base_dir = Path("../data")
img_train_dir = base_dir / "images" / "train"
img_val_dir = base_dir / "images" / "val"
img_train_dir.mkdir(parents=True, exist_ok=True)
img_val_dir.mkdir(parents=True, exist_ok=True)

# 1. Load JSONs to know exactly which image belongs where
train_json = next(Path("../..").rglob("instances_train_trashcan.json"), None)
val_json = next(Path("../..").rglob("instances_val_trashcan.json"), None)

with open(train_json, "r") as f:
    train_data = json.load(f)
train_filenames = {os.path.basename(img['file_name']) for img in train_data['images']}

with open(val_json, "r") as f:
    val_data = json.load(f)
val_filenames = {os.path.basename(img['file_name']) for img in val_data['images']}

# 2. Find ALL jpgs anywhere in the data folder
all_jpgs = list(base_dir.rglob("*.jpg"))
print(f"Found {len(all_jpgs)} total images! Moving them now...")

moved_train = 0
moved_val = 0

# 3. Move each file to its exact, perfect destination
for jpg_path in all_jpgs:
    file_name = jpg_path.name
    if file_name in train_filenames:
        dest = img_train_dir / file_name
        if jpg_path != dest:  # Only move if it's not already in the perfect spot
            shutil.move(str(jpg_path), str(dest))
        moved_train += 1
    elif file_name in val_filenames:
        dest = img_val_dir / file_name
        if jpg_path != dest:
            shutil.move(str(jpg_path), str(dest))
        moved_val += 1

print("\n--- FIX REPORT ---")
print(f"Successfully secured {moved_train} images in {img_train_dir}")
print(f"Successfully secured {moved_val} images in {img_val_dir}")
print("YOLO dataset structure is now 100% perfect. Ready for Git Push!")

In [ ]:
# 1. Mount Google Drive to access the zip file
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

# 2. Define exactly where the zip is, and where we want to unpack it
# (Using the exact folder name from your Google Drive)
zip_path = '/content/drive/MyDrive/Underwater_ai_drone/yolo_dataset.zip'
extract_path = '/content/dataset'

# 3. Extract to Colab's high-speed local storage
print("\nExtracting dataset... This will take a minute or two.")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Extraction complete!")

# 4. Install Ultralytics (YOLOv8 framework)
!pip install ultralytics
import ultralytics

# 5. Verify the GPU is active
print("\n--- ENVIRONMENT CHECK ---")
ultralytics.checks()